In [ ]:
"""B1 baseline (own pipeline) -- label-free retrieval + FPNet + library similarity + GLACIER/ICEBERG + fixed fusion.
Sources embedded from experiments/b1_baseline (sha256 b1_common bc657100b794, b1_pipeline 9ab3a4a57b5b)."""
import os, sys, glob, subprocess, time, json
T0 = time.time()
os.makedirs('/kaggle/working/b1', exist_ok=True)
SOURCES = {"b1_common.py": "\"\"\"B1 shared definitions: adduct -> neutral mass, electron-mass convention, scorer identity, entropy similarity.\n\nIon m/z = M + sum(adduct atoms) -/+ electron (positive: minus m_e, negative: plus m_e). enveda-180 positive-mode m/z carry a +m_e bias\n(no electron removed) and are corrected by the caller (subtract ELECTRON before using these formulas).\n\"\"\"\nimport numpy as np\n\nELECTRON = 0.00054858\nH, C, N, O, Na, K, Cl = 1.00782503, 12.0, 14.0030740, 15.9949146, 22.9897693, 38.9637067, 34.9688527\nH2O = 2 * H + O\n# neutral mass M = prec + ADDUCT_SHIFT[adduct]\nADDUCT_SHIFT = {\n    \"[M+H]+\": -(H - ELECTRON),\n    \"[M+NH4]+\": -(N + 4 * H - ELECTRON),\n    \"[M+Na]+\": -(Na - ELECTRON),\n    \"[M+K]+\": -(K - ELECTRON),\n    \"[M-H2O+H]+\": H2O - (H - ELECTRON),\n    \"[M-2H2O+H]+\": 2 * H2O - (H - ELECTRON),\n    \"[M-H]-\": (H - ELECTRON),\n    \"[M-H2O-H]-\": H2O + (H - ELECTRON),\n    \"[M+CH2O2-H]-\": -(C + 2 * H + 2 * O) + (H - ELECTRON),\n    \"[M+Cl]-\": -(Cl + ELECTRON),\n}\nTEST_ADDUCTS = list(ADDUCT_SHIFT)\nPOSITIVE = {a for a in ADDUCT_SHIFT if a.endswith(\"+\")}\n\n\ndef neutral_mass(prec, adduct):\n    s = ADDUCT_SHIFT.get(adduct)\n    return None if s is None else float(prec) + s\n\n\n# ----------------------------------------------------------------------------------------------- spectral entropy similarity\ndef clean_peaks(mz, it, prec=None, floor=0.001, top=512, prec_margin=1.5):\n    \"\"\"relative-intensity filter, drop peaks at/above the precursor (minus margin), keep the top-N, L1-normalise.\"\"\"\n    mz = np.asarray(mz, np.float64); it = np.asarray(it, np.float64)\n    ok = np.isfinite(mz) & np.isfinite(it) & (it > 0)\n    if prec is not None: ok &= mz < float(prec) - prec_margin\n    mz, it = mz[ok], it[ok]\n    if len(it) == 0: return mz, it\n    it = it / it.max(); keep = it >= floor; mz, it = mz[keep], it[keep]\n    if len(it) > top:\n        idx = np.argpartition(-it, top - 1)[:top]; mz, it = mz[idx], it[idx]\n    o = np.argsort(mz); mz, it = mz[o], it[o]\n    return mz, it / it.sum()\n\n\ndef entropy_weight(it):\n    \"\"\"Li et al. 2021 entropy weighting: S < 3 -> intensity^(0.25 + 0.25 S), renormalised.\"\"\"\n    p = it / it.sum(); s = float(-(p[p > 0] * np.log(p[p > 0])).sum())\n    if s < 3.0:\n        w = 0.25 + 0.25 * s; p = p ** w; p = p / p.sum()\n    return p\n\n\ndef merge_tolerance(mz_a, ia, mz_b, ib, tol_da=0.01, tol_ppm=20.0):\n    \"\"\"align two sorted peak lists: greedy nearest matching within max(tol_da, tol_ppm) -> aligned intensity vectors.\"\"\"\n    i = j = 0; a_out, b_out = [], []\n    while i < len(mz_a) and j < len(mz_b):\n        tol = max(tol_da, mz_a[i] * tol_ppm * 1e-6); d = mz_a[i] - mz_b[j]\n        if abs(d) <= tol:\n            a_out.append(ia[i]); b_out.append(ib[j]); i += 1; j += 1\n        elif d < 0:\n            a_out.append(ia[i]); b_out.append(0.0); i += 1\n        else:\n            a_out.append(0.0); b_out.append(ib[j]); j += 1\n    while i < len(mz_a): a_out.append(ia[i]); b_out.append(0.0); i += 1\n    while j < len(mz_b): a_out.append(0.0); b_out.append(ib[j]); j += 1\n    return np.asarray(a_out), np.asarray(b_out)\n\n\ndef entropy_similarity(mz_a, ia, mz_b, ib, tol_da=0.01, tol_ppm=20.0):\n    \"\"\"spectral entropy similarity in [0, 1] of two cleaned (sorted, L1-normalised) spectra.\"\"\"\n    if len(ia) == 0 or len(ib) == 0: return 0.0\n    pa, pb = entropy_weight(ia), entropy_weight(ib)\n    a, b = merge_tolerance(mz_a, pa, mz_b, pb, tol_da, tol_ppm)\n    m = (a + b) / 2.0\n    def ent(p):\n        p = p[p > 0]; return float(-(p * np.log(p)).sum())\n    return max(0.0, min(1.0, 1.0 - (2 * ent(m) - ent(a) - ent(b)) / np.log(4)))\n", "b1_pipeline.py": "\"\"\"B1 pipeline -- label-free MS/MS -> structure identification for a test parquet, end to end, locally or on Kaggle.\n\nStages (all label-free; see docs/b1_baseline_design.md):\n  1. test spectra per molecule (10 adducts, both polarities; electron-mass convention of b1_common)\n  2. train spectra store built from train.parquet (library search references) -- in memory on Kaggle, mmap locally\n  3. candidates: neutral-mass windows (+-10 ppm per observed adduct) over the PubChem tier + the v2 pool (train structures + COCONUT),\n     parsed once in worker processes (raw InChIKey14 + FPNet f.z over the fold-model ensemble), merged on the raw key with provenance\n  4. library similarity (spectral entropy, same polarity) for candidates that are train structures\n  5. documentation priors (popularity dataset, tier / pool rows)\n  6. shortlist = top-K by FP, top-K by z(FP)+0.15 pop, top-20 by library similarity; official scorer keys; duplicates merged\n  7. GLACIER / ICEBERG on the shortlists (budgeted; molecules with the weakest library match first)\n  8. fusion: S = z(FP) + W_GL z(GL) + W_ICE z(ICE) + LAM pop; library-priority when the best match >= TAU and beats the runner-up by MARGIN\n  9. top-25 by official identity -> submission.csv; stage statistics -> b1_run_manifest.json\nUsage (local): python -X utf8 b1_pipeline.py --test data/raw/test.parquet --out <dir> [--limit N] [--no_forward]\nOn Kaggle the paths are resolved under /kaggle/input/** (same datasets as the public E0 kernel) and /kaggle/working.\n\"\"\"\nimport os as _os\nfor _v in (\"OMP_NUM_THREADS\", \"OPENBLAS_NUM_THREADS\", \"MKL_NUM_THREADS\", \"NUMBA_NUM_THREADS\"): _os.environ.setdefault(_v, \"1\")   # one BLAS thread per process: 16 workers x 64 OpenBLAS threads made the Spark 30x slower\nimport argparse, glob, json, os, subprocess, sys, time\nfrom collections import Counter\nfrom multiprocessing import Pool\nimport numpy as np, pandas as pd, polars as pl\n\nHERE = os.path.dirname(os.path.abspath(__file__)); sys.path.insert(0, HERE)\nfrom b1_common import ADDUCT_SHIFT, ELECTRON, clean_peaks, entropy_similarity  # noqa: E402\n\nKAGGLE = os.path.exists(\"/kaggle/input\")\nPPM, K_SHORT, K_LIB, LIB_MIN, MAX_REF, MAX_Q = 10.0, 100, 20, 0.2, 24, 8      # MAX_Q 8 with adduct round-robin = the evaluation's query policy\nSHORTLIST_LAM = 0.15   # prior weight for the shortlist (as evaluated); LAM below is the fusion weight\nMAX_HEAVY_FORWARD = 100   # structures above this are not sent to the forward models\nDUP_SIM = 0.98            # a reference at the same precursor this similar to a query spectrum is treated as the same acquisition (re-deposit) and not used\nW_GL, W_ICE, LAM, TAU, MARGIN = 2.0, 1.0, 0.5, 2.0, 0.2   # LAM 0.5 (B1 eval: better than 0.15 on A, B and the C1 set); TAU 2.0 disables the library-priority rule until the deduplicated C1 evaluation supports it\nICE_BUDGET, GL_BUDGET = 5400, 4000\n\n\ndef find(pattern, root=\"/kaggle/input\"):\n    hits = sorted(glob.glob(os.path.join(root, \"**\", pattern), recursive=True))\n    if not hits: raise FileNotFoundError(pattern)\n    return hits[0]\n\n\ndef paths(args):\n    if KAGGLE:\n        def first(*pats):\n            for pat in pats:\n                try: return find(pat)\n                except FileNotFoundError: pass\n            raise FileNotFoundError(pats)\n        pop = os.path.dirname(first(\"pc_lsid.npy\"))\n        tier = os.path.dirname(first(\"casmi26-pubchem-tier/pc_mass.npy\", \"pubchem_tier/pc_mass.npy\", \"pc_mass.npy\"))\n        return dict(train=\"/kaggle/input/enveda-CASMI26-molecule-id-mass-spectra/train.parquet\", tier=tier, pop=pop,\n                    pool=first(\"casmi26-v2-pool/pool_meta.parquet\", \"pool_meta.parquet\"), fold_safe=os.path.dirname(first(\"fpnet_fold0.pt\")),\n                    v3=os.path.dirname(os.path.dirname(first(\"casmi26-v3-models/code/casmi/fpnet.py\", \"v3_models/code/casmi/fpnet.py\"))),\n                    glacier=os.path.dirname(first(\"gl_runner.py\")), iceberg=os.path.dirname(first(\"ice_runner.py\")), work=\"/kaggle/working\", store=None, runner_python=None)\n    R = os.environ.get(\"CASMI_ROOT\", r\"C:\\kaggle\\CASMI\" if os.name == \"nt\" else os.path.expanduser(\"~/CASMI2026\"))\n    runner_py = f\"{R}/.venv-ice/Scripts/python.exe\" if os.name == \"nt\" else f\"{R}/.venv-ice/bin/python\"\n    return dict(train=f\"{R}/data/raw/train.parquet\", tier=f\"{R}/data/external/pubchem_tier\", pop=f\"{R}/data/external/pubchem_popularity\", pool=f\"{R}/data/external/v2_pool/pool_meta.parquet\",\n                fold_safe=f\"{R}/data/external/fold_safe_fpnet\", v3=f\"{R}/data/external/v3_models/code\", glacier=f\"{R}/data/external/glacier\", iceberg=f\"{R}/data/external/iceberg\",\n                work=args.out, store=f\"{R}/data/processed/spectra_store\", runner_python=runner_py)\n\n\n# ------------------------------------------------------------------------------------------------------------ workers\ndef _init(v3, fs):\n    global chem_w, bits_w, Chem, TAUT\n    sys.path.insert(0, v3)\n    from casmi import chem as chem_w\n    from rdkit import Chem, RDLogger\n    from rdkit.Chem.MolStandardize import rdMolStandardize\n    RDLogger.DisableLog(\"rdApp.*\"); TAUT = rdMolStandardize.TautomerEnumerator(); bits_w = np.load(f\"{fs}/fp_bits.npy\")\n\n\ndef _feat(args):\n    smiles, zlog = args\n    keys = [None] * len(smiles); sc = np.full(len(smiles), np.nan, np.float32)\n    for i, s in enumerate(smiles):\n        m = Chem.MolFromSmiles(s)\n        if m is None: continue\n        try: keys[i] = Chem.MolToInchiKey(m)[:14]\n        except Exception: continue\n        f = chem_w.raw_fingerprint(s)\n        if f is not None: sc[i] = float(f[bits_w].astype(np.float32) @ zlog)\n    return keys, sc\n\n\ndef _rawkey(smiles):\n    out = []\n    for s in smiles:\n        try:\n            m = Chem.MolFromSmiles(s); out.append(Chem.MolToInchiKey(m)[:14] if m is not None else None)\n        except Exception:\n            out.append(None)\n    return out\n\n\ndef _okey(smiles):\n    out = []\n    for s in smiles:\n        try:\n            m = Chem.MolFromSmiles(s); Chem.RemoveStereochemistry(m); out.append(Chem.MolToInchiKey(TAUT.Canonicalize(m))[:14])\n        except Exception:\n            out.append(None)\n    return out\n\n\n# ------------------------------------------------------------------------------------------------------------ forward-model site (Kaggle)\ndef _compatible_rdkit_wheel(pkg):\n    \"\"\"RDKit 2025.3.6 wheel compatible with this interpreter (bundled in the ICEBERG package or attached as a dataset).\"\"\"\n    from pip._vendor.packaging.tags import sys_tags\n    from pip._vendor.packaging.utils import parse_wheel_filename\n    compatible = set(sys_tags())\n    for path in sorted(glob.glob(os.path.join(pkg, \"wheels\", \"rdkit-2025.3.6-*.whl*\"))) + sorted(glob.glob(\"/kaggle/input/**/rdkit-2025.3.6-*.whl\", recursive=True)):\n        name = os.path.basename(path).removesuffix(\".ice\")\n        _, version, _, tags = parse_wheel_filename(name)\n        if str(version) == \"2025.3.6\" and compatible.intersection(tags): return path\n    raise FileNotFoundError(\"no compatible RDKit 2025.3.6 wheel for the ICEBERG/GLACIER site\")\n\n\ndef preseed_site(pkg, site):\n    \"\"\"install the ICEBERG wheels (+ a compatible RDKit) into `site` with pip --target and write the runner's marker (same procedure as E0).\"\"\"\n    import shutil\n    srcs = sorted(glob.glob(os.path.join(pkg, \"wheels\", \"*.whl.ice\")) + glob.glob(os.path.join(pkg, \"wheels\", \"*.whl\")))\n    keep = [w for w in srcs if not os.path.basename(w).lower().startswith(\"rdkit\")] + [_compatible_rdkit_wheel(pkg)]\n    os.makedirs(site, exist_ok=True); wdir = os.path.join(site, \".wheels\"); os.makedirs(wdir, exist_ok=True); whl = []\n    for w in keep:\n        dst = os.path.join(wdir, os.path.basename(w)[:-4] if w.endswith(\".ice\") else os.path.basename(w)); shutil.copyfile(w, dst); whl.append(dst)\n    r = subprocess.run([sys.executable, \"-m\", \"pip\", \"install\", \"-q\", \"--no-index\", \"--no-deps\", \"--no-compile\", \"--disable-pip-version-check\", \"--upgrade\", \"--target\", site] + whl,\n                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)\n    print(\"[b1] site preseed rc\", r.returncode, (r.stdout or \"\")[-300:], flush=True)\n    if r.returncode != 0: raise RuntimeError(\"site preseed failed: \" + (r.stdout or \"\")[-500:])\n    shutil.rmtree(wdir, ignore_errors=True)\n    open(os.path.join(site, \".ice_site_ok\"), \"w\").write(chr(10).join(os.path.basename(w) for w in srcs))\n\n\ndef heavy_ok(smiles):\n    try:\n        from rdkit import Chem\n        m = Chem.MolFromSmiles(smiles); return m is not None and m.GetNumHeavyAtoms() <= MAX_HEAVY_FORWARD\n    except Exception:\n        return False\n\n\n# ------------------------------------------------------------------------------------------------------------ data\ndef load_test(path, limit=0):\n    t = pl.read_parquet(path)\n    cols = [\"molecule_id\", \"ms2_mzs\", \"ms2_normalized_intensities\", \"precursor_mz\", \"adduct\", \"ionization_mode\", \"instrument_type\", \"collision_energy_ev\", \"collision_energy_orig\", \"collision_energy_orig_units\"]\n    t = t.select([c for c in cols if c in t.columns])\n    mols = {}\n    for r in t.iter_rows(named=True):\n        ce = r.get(\"collision_energy_ev\"); ce = [float(x) for x in ce] if ce is not None and len(ce) else None\n        mols.setdefault(str(r[\"molecule_id\"]), []).append(dict(mz=np.asarray(r[\"ms2_mzs\"], np.float64), it=np.asarray(r[\"ms2_normalized_intensities\"], np.float64), prec=float(r[\"precursor_mz\"]),\n                                                                 adduct=r[\"adduct\"], mode=r[\"ionization_mode\"], instrument=r.get(\"instrument_type\") or \"\", ce=float(np.mean(ce)) if ce else None,\n                                                                 ce_ev=ce, ce_orig=r.get(\"collision_energy_orig\"), ce_units=r.get(\"collision_energy_orig_units\")))\n    ids = list(mols)\n    if limit: ids = ids[:limit]\n    return {m: mols[m] for m in ids}, t[\"molecule_id\"].unique().to_list()\n\n\ndef build_store(train_parquet, scorer_keys_parquet=None):\n    \"\"\"flat spectra arrays + meta from train.parquet (in memory); returns dict.\"\"\"\n    import pyarrow.parquet as pq\n    pf = pq.ParquetFile(train_parquet); mz_parts, it_parts, lens, metas = [], [], [], []\n    for b in pf.iter_batches(batch_size=100_000, columns=[\"inchikey14\", \"ingest_lib\", \"adduct\", \"ionization_mode\", \"precursor_mz\", \"collision_energy_ev\", \"ms2_mzs\", \"ms2_normalized_intensities\"]):\n        mz = b.column(\"ms2_mzs\"); it = b.column(\"ms2_normalized_intensities\")\n        for cm, ci in zip(mz.chunks if hasattr(mz, \"chunks\") else [mz], it.chunks if hasattr(it, \"chunks\") else [it]):\n            lens.append(np.diff(np.asarray(cm.offsets)).astype(np.int64)); mz_parts.append(np.asarray(cm.flatten(), np.float32)); it_parts.append(np.asarray(ci.flatten(), np.float32))\n        tb = pl.from_arrow(b.select([\"inchikey14\", \"ingest_lib\", \"adduct\", \"ionization_mode\", \"precursor_mz\", \"collision_energy_ev\"]))\n        metas.append(tb.with_columns(pl.col(\"collision_energy_ev\").list.mean().alias(\"ce_mean\")).drop(\"collision_energy_ev\"))\n    lens = np.concatenate(lens); off = np.zeros(len(lens) + 1, np.int64); np.cumsum(lens, out=off[1:])\n    meta = pl.concat(metas).with_row_index(\"row\")\n    return dict(mz=np.concatenate(mz_parts), it=np.concatenate(it_parts), off=off, meta=meta)\n\n\ndef load_store(store_dir):\n    return dict(mz=np.load(f\"{store_dir}/spec_mz.npy\", mmap_mode=\"r\"), it=np.load(f\"{store_dir}/spec_int.npy\", mmap_mode=\"r\"), off=np.load(f\"{store_dir}/spec_off.npy\"),\n                meta=pl.read_parquet(f\"{store_dir}/spec_meta.parquet\"))\n\n\ndef official_keys_of_train(store, pool):\n    \"\"\"map train structures (inchikey14) -> official scorer key through the pool (pool keys are official; src 1/3 rows carry train_sid) --\n    falls back to canonicalising structures absent from the pool.\"\"\"\n    return None\n\n\n# ------------------------------------------------------------------------------------------------------------ main\ndef main():\n    ap = argparse.ArgumentParser(); ap.add_argument(\"--test\", default=None); ap.add_argument(\"--out\", default=\"experiments/b1_baseline/pipeline_local\"); ap.add_argument(\"--limit\", type=int, default=0)\n    ap.add_argument(\"--no_forward\", action=\"store_true\"); ap.add_argument(\"--workers\", type=int, default=0); ap.add_argument(\"--folds\", default=\"0\")\n    ap.add_argument(\"--rank_model\", default=None, help=\"LightGBM LambdaRank model file (default: lambdarank.txt next to this script; absent -> fixed rule)\")\n    ap.add_argument(\"--exclude_rows\", default=None, help=\"evaluation: JSON list of spectra-store rows never used as references (the query acquisitions) -> class-1 proxy\")\n    ap.add_argument(\"--exclude_identities\", default=None, help=\"evaluation: JSON list of official keys whose references are removed and whose train membership is hidden -> class-2 proxy\")\n    ap.add_argument(\"--solution\", default=None, help=\"evaluation: CSV with molecule_id,smiles; scores the submission with the official scorer replica\")\n    args = ap.parse_args(); P = paths(args); os.makedirs(P[\"work\"], exist_ok=True); T0 = time.time(); log = lambda *x: print(f\"[b1 {time.time()-T0:7.0f}s]\", *x, flush=True)\n    test_path = args.test or (\"/kaggle/input/enveda-CASMI26-molecule-id-mass-spectra/test.parquet\" if KAGGLE else os.path.join(os.path.dirname(P[\"train\"]), \"test.parquet\"))\n    workers = args.workers or max(2, (os.cpu_count() or 4) - 1)\n    import torch\n    sys.path.insert(0, P[\"v3\"])\n    from casmi import fpnet, chem\n    from casmi.spectra import merge_spectra\n    from rdkit import Chem, RDLogger\n    from rdkit.Chem.MolStandardize import rdMolStandardize\n    RDLogger.DisableLog(\"rdApp.*\"); TAUT = rdMolStandardize.TautomerEnumerator()\n    dev = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n    mols, all_ids = load_test(test_path, args.limit); log(f\"test: {len(mols)} molecules of {len(all_ids)}; {sum(len(v) for v in mols.values())} spectra\")\n    # ---- stores\n    store = load_store(P[\"store\"]) if P[\"store\"] and os.path.exists(P[\"store\"]) else build_store(P[\"train\"])\n    meta = store[\"meta\"]; S_MZ, S_IT, S_OFF = store[\"mz\"], store[\"it\"], store[\"off\"]\n    lib_of_row = meta[\"ingest_lib\"].to_numpy(); add_of_row = meta[\"adduct\"].to_numpy(); mode_of_row = meta[\"ionization_mode\"].to_numpy(); prec_of_row = meta[\"precursor_mz\"].to_numpy()\n    ik_rows = {k: np.asarray(r, np.int64) for k, r in meta.group_by(\"inchikey14\").agg(pl.col(\"row\")).iter_rows()}\n    log(f\"store: {meta.height:,} spectra, {len(ik_rows):,} train structures\")\n    pool = pl.read_parquet(P[\"pool\"]).select(\"key\", \"smiles\", \"mass\", \"src\", \"pid\")\n    tmass = np.load(f\"{P['tier']}/pc_mass.npy\", mmap_mode=\"r\"); toff = np.load(f\"{P['tier']}/pc_off.npy\", mmap_mode=\"r\"); tbuf = np.load(f\"{P['tier']}/pc_smiles.npy\", mmap_mode=\"r\")\n    PC = {n: np.load(f\"{P['pop']}/pc_{n}.npy\", mmap_mode=\"r\") for n in (\"lsid\", \"lpmid\", \"lsynonyms\", \"np\", \"src_mask\", \"lotus_refs\")}\n    PL = {n: np.load(f\"{P['pop']}/pool_{n}.npy\") for n in (\"lsid\", \"lpmid\", \"lsynonyms\", \"np\", \"src_mask\", \"lotus_refs\")}\n    assert all(v.shape[0] == tmass.shape[0] for v in PC.values()), 'popularity arrays are not aligned to the tier'\n    assert all(v.shape[0] == pool.height for v in PL.values()), 'pool popularity arrays are not aligned to the pool'\n    log(f\"paths: {json.dumps({k: str(v) for k, v in P.items()})}\")\n    CMASS = CSMI = None\n    try:\n        import pickle\n        cdir = os.path.dirname(find(\"coco_mass.npy\")) if KAGGLE else os.path.join(os.path.dirname(os.path.dirname(P[\"pool\"])), \"coconut\")\n        if os.path.exists(f\"{cdir}/coco_mass.npy\"):\n            CMASS = np.load(f\"{cdir}/coco_mass.npy\"); CSMI = np.asarray(pickle.load(open(f\"{cdir}/coco_meta.pkl\", \"rb\"))[\"smiles\"]); log(f\"COCONUT candidate store: {len(CMASS):,} structures\")\n    except Exception as e:\n        CMASS = CSMI = None; log(\"COCONUT store not used:\", repr(e))\n    order = np.argsort(pool[\"mass\"].to_numpy(), kind=\"stable\")\n    pmass = pool[\"mass\"].to_numpy()[order]; psmi = pool[\"smiles\"].to_numpy()[order]; psrc = pool[\"src\"].to_numpy()[order]; ppid = pool[\"pid\"].to_numpy()[order]; pkey = pool[\"key\"].to_numpy()[order]\n    folds = [int(x) for x in args.folds.split(\",\")]\n    models = [fpnet.load_model(f\"{P['fold_safe']}/fpnet_fold{k}.pt\", dev) for k in folds]; dtype = fpnet.autocast_dtype(dev)\n    log(f\"FPNet folds {folds} loaded on {dev}; tier rows {tmass.shape[0]:,}; pool {pool.height:,}\")\n\n    def spectrum(row, shift=0.0):\n        s, e = int(S_OFF[row]), int(S_OFF[row + 1]); return np.asarray(S_MZ[s:e], np.float64) - shift, np.asarray(S_IT[s:e], np.float64)\n\n    def tier_window(lo_m, hi_m):\n        lo = int(np.searchsorted(tmass, lo_m, \"left\")); hi = int(np.searchsorted(tmass, hi_m, \"right\"))\n        if hi <= lo: return [], np.zeros(0, np.int64), np.zeros(0)\n        o = np.asarray(toff[lo:hi + 1]).astype(np.int64); raw = bytes(tbuf[o[0]:o[-1]]); base = o[0]\n        return [raw[o[i] - base:o[i + 1] - base].decode(\"ascii\") for i in range(hi - lo)], np.arange(lo, hi), np.asarray(tmass[lo:hi], np.float64)\n\n    stats, feats, runner, ref_cache, molinfo = [], {}, [], {}, {}\n    with Pool(workers, initializer=_init, initargs=(P[\"v3\"], P[\"fold_safe\"])) as wp, torch.no_grad():\n        # (official key -> reference rows) for train structures: scorer keys from the store meta when available (local), else raw\n        # InChIKey14 of the pool's train SMILES computed in the workers and matched to the store's inchikey14\n        if \"scorer_ik14\" in meta.columns:\n            key_rows = {k: np.asarray(r, np.int64) for k, r in meta.filter(pl.col(\"scorer_ik14\").is_not_null()).group_by(\"scorer_ik14\").agg(pl.col(\"row\")).iter_rows()}\n        else:\n            tr = pool.filter(pl.col(\"src\").is_in([1, 3])); tsmi = tr[\"smiles\"].to_list(); tkey = tr[\"key\"].to_list()\n            rk = [k for rr_ in wp.map(_rawkey, [tsmi[i:i + 2000] for i in range(0, len(tsmi), 2000)]) for k in rr_]\n            ik_to_key = {ik: key for ik, key in zip(rk, tkey) if ik}\n            key_rows = {}\n            for ik, rows in ik_rows.items():\n                k = ik_to_key.get(ik)\n                if k: key_rows.setdefault(k, []).append(rows)\n            key_rows = {k: np.concatenate(v) for k, v in key_rows.items()}\n        log(f\"library index: {len(key_rows):,} official identities with reference spectra\")\n        EXCL_ROWS = set(json.load(open(args.exclude_rows))) if args.exclude_rows else set()\n        EXCL_IDS = set(json.load(open(args.exclude_identities))) if args.exclude_identities else set()\n        if EXCL_ROWS or EXCL_IDS: log(f\"EVALUATION MODE: {len(EXCL_ROWS)} reference rows and {len(EXCL_IDS)} identities masked\")\n        for n_i, (mid, spectra) in enumerate(mols.items()):\n            tm0 = time.time()\n            known = [s for s in spectra if s[\"adduct\"] in ADDUCT_SHIFT]\n            by_add = {}\n            for s in sorted(known, key=lambda s: (s[\"adduct\"] != \"[M+H]+\", s[\"adduct\"], s[\"ce\"] if s[\"ce\"] is not None else 0.0)):\n                by_add.setdefault(s[\"adduct\"], []).append(s)\n            q = []\n            while len(q) < MAX_Q and any(by_add.values()):          # adduct round-robin (same policy as b1_features.py)\n                for ad in list(by_add):\n                    if by_add[ad] and len(q) < MAX_Q: q.append(by_add[ad].pop(0))\n            q = [dict(s, clean=clean_peaks(s[\"mz\"], s[\"it\"], s[\"prec\"])) for s in q]\n            if not q:                                         # unknown adducts only: fall back to [M+H]+ / [M-H]- by polarity\n                q = [dict(s, adduct=(\"[M+H]+\" if s[\"mode\"] == \"positive\" else \"[M-H]-\"), clean=clean_peaks(s[\"mz\"], s[\"it\"], s[\"prec\"])) for s in spectra][:MAX_Q]\n            M_by_add = {}\n            for s in q: M_by_add.setdefault(s[\"adduct\"], []).append(s[\"prec\"] + ADDUCT_SHIFT[s[\"adduct\"]])\n            M_by_add = {k: float(np.median(v)) for k, v in M_by_add.items()}\n            src = {}   # smiles -> provenance (de-duplicated across the adduct windows before fingerprinting)\n            for M in M_by_add.values():\n                tol = M * PPM * 1e-6; ts, trows, tms = tier_window(M - tol, M + tol)\n                for s_, tr_, m_ in zip(ts, trows, tms):\n                    d = src.setdefault(s_, dict(train=False, coco=False, trow=-1, pid=-1, pkey=None, mass=float(m_)))\n                    if d[\"trow\"] < 0: d[\"trow\"] = int(tr_)\n                lo = int(np.searchsorted(pmass, M - tol, \"left\")); hi = int(np.searchsorted(pmass, M + tol, \"right\"))\n                for s_, sr_, pi_, pk_, m_ in zip(psmi[lo:hi], psrc[lo:hi], ppid[lo:hi], pkey[lo:hi], pmass[lo:hi]):\n                    d = src.setdefault(str(s_), dict(train=False, coco=False, trow=-1, pid=-1, pkey=None, mass=float(m_)))\n                    d[\"train\"] |= int(sr_) in (1, 3); d[\"coco\"] |= int(sr_) in (2, 3)\n                    if d[\"pid\"] < 0: d[\"pid\"] = int(pi_); d[\"pkey\"] = str(pk_)\n                if CMASS is not None:\n                    clo = int(np.searchsorted(CMASS, M - tol, \"left\")); chi = int(np.searchsorted(CMASS, M + tol, \"right\"))\n                    for s_, m_ in zip(CSMI[clo:chi], CMASS[clo:chi]):\n                        d = src.setdefault(str(s_), dict(train=False, coco=False, trow=-1, pid=-1, pkey=None, mass=float(m_))); d[\"coco\"] = True\n            smiles = list(src)\n            src_train = [src[s_][\"train\"] for s_ in smiles]; src_coco = [src[s_][\"coco\"] for s_ in smiles]; tier_row = [src[s_][\"trow\"] for s_ in smiles]\n            pool_pid = [src[s_][\"pid\"] for s_ in smiles]; pool_key = [src[s_][\"pkey\"] for s_ in smiles]; cmass = [src[s_][\"mass\"] for s_ in smiles]\n            # FPNet logits (polarity groups, fold ensemble)\n            zl, wsum = None, 0.0\n            for mode in (\"positive\", \"negative\"):\n                grp = [s for s in q if s[\"mode\"] == mode]\n                if not grp: continue\n                pairs = [(s[\"mz\"], s[\"it\"]) for s in grp]; mz, it = pairs[0] if len(pairs) == 1 else merge_spectra(pairs)\n                prec = float(np.median([s[\"prec\"] for s in grp])); ces = [s[\"ce\"] for s in grp if s[\"ce\"] is not None]\n                ad = Counter(s[\"adduct\"] for s in grp).most_common(1)[0][0]; pm, pi = fpnet.prep_peaks(mz, it, prec)\n                inp = fpnet.collate([dict(mz=pm, it=pi, prec=prec, adduct_ix=chem.adduct_index(ad), ce=float(np.mean(ces)) if ces else 0.0, ce_known=1.0 if ces else 0.0,\n                                          n_merged=min(len(grp), 8), mode=1.0 if mode == \"positive\" else -1.0)], dev)\n                with torch.autocast(dev, dtype=dtype, enabled=(dev == \"cuda\")):\n                    zz = torch.stack([m(*inp)[0].float()[0] for m in models]).mean(0).cpu().numpy().astype(np.float32)\n                zl = zz * len(grp) if zl is None else zl + zz * len(grp); wsum += len(grp)\n            zlog = (zl / wsum).astype(np.float32)\n            chunks = [smiles[i:i + 400] for i in range(0, len(smiles), 400)]\n            res = wp.map(_feat, [(c, zlog) for c in chunks]) if chunks else []\n            raw_keys = [k for rr_ in res for k in rr_[0]]; fp = np.concatenate([rr_[1] for rr_ in res]) if res else np.zeros(0, np.float32)\n            rep = {}\n            for i, k in enumerate(raw_keys):\n                if k is None: continue\n                if k not in rep: rep[k] = dict(i=i, train=src_train[i], coco=src_coco[i], trow=tier_row[i], pid=pool_pid[i], pkey=pool_key[i], mass=cmass[i], fp=float(fp[i]) if np.isfinite(fp[i]) else -np.inf)\n                else:\n                    p = rep[k]; p[\"train\"] |= src_train[i]; p[\"coco\"] |= src_coco[i]\n                    if p[\"trow\"] < 0: p[\"trow\"] = tier_row[i]\n                    if p[\"pid\"] < 0: p[\"pid\"] = pool_pid[i]; p[\"pkey\"] = pool_key[i]\n                    if np.isfinite(fp[i]) and fp[i] > p[\"fp\"]: p[\"fp\"] = float(fp[i])\n            if EXCL_IDS:\n                for k in list(rep):\n                    if rep[k][\"pkey\"] in EXCL_IDS:\n                        if rep[k][\"trow\"] < 0 and not rep[k][\"coco\"]: del rep[k]      # a library-only structure would not exist as a candidate for a novel molecule\n                        else: rep[k][\"train\"] = False\n            keys = list(rep); n_u = len(keys)\n            if n_u == 0:\n                stats.append(dict(molecule=mid, n_query=len(q), n_unique=0, seconds=time.time() - tm0)); feats[mid] = []; continue\n            fpu = np.array([rep[k][\"fp\"] for k in keys]); fpu[~np.isfinite(fpu)] = np.nan\n            trow = np.array([rep[k][\"trow\"] for k in keys]); pid = np.array([rep[k][\"pid\"] for k in keys]); in_train = np.array([rep[k][\"train\"] for k in keys]); in_coco = np.array([rep[k][\"coco\"] for k in keys])\n            cm_ = np.array([rep[k][\"mass\"] for k in keys], float)\n            comp = {n: np.zeros(n_u, np.float32) for n in PC}; ht = trow >= 0; hp = (~ht) & (pid >= 0)\n            for n in comp:\n                if ht.any(): comp[n][ht] = np.asarray(PC[n][np.sort(trow[ht])], dtype=np.float32)[np.argsort(np.argsort(trow[ht]))]\n                if hp.any(): comp[n][hp] = np.asarray(PL[n][pid[hp]], dtype=np.float32)\n            comp[\"np\"][in_coco] = np.maximum(comp[\"np\"][in_coco], 1.0)\n            pop = comp[\"lsid\"] + comp[\"lpmid\"] + 0.5 * comp[\"lsynonyms\"] + 1.0 * (comp[\"np\"] != 0) + 0.5 * ((comp[\"src_mask\"].astype(np.int64) & 1) != 0) + 0.5 * np.log1p(np.maximum(comp[\"lotus_refs\"], 0))\n            # library similarity\n            lib = np.zeros(n_u); lib_same = np.zeros(n_u); nref = np.zeros(n_u, int)\n            for ci, k in enumerate(keys):\n                pk = rep[k][\"pkey\"]\n                if not rep[k][\"train\"] or pk is None or pk not in key_rows or pk in EXCL_IDS: continue\n                rrows = [int(x) for x in key_rows[pk] if x not in EXCL_ROWS]\n                kept = []\n                for x in rrows:                                       # duplicate acquisitions of a query spectrum are not independent references\n                    if x not in ref_cache:\n                        sh = ELECTRON if (lib_of_row[x] == \"enveda-180\" and mode_of_row[x] == \"positive\") else 0.0\n                        mz, it = spectrum(x, sh); ref_cache[x] = (clean_peaks(mz, it, float(prec_of_row[x]) - sh), float(prec_of_row[x]) - sh)\n                    (rmz, rit), rprec = ref_cache[x]\n                    dup = False\n                    for s in q:\n                        if abs(s[\"prec\"] - rprec) < 0.005 and s[\"mode\"] == mode_of_row[x]:\n                            qmz, qit = s[\"clean\"]\n                            if (s[\"adduct\"] == add_of_row[x] and len(qmz) == len(rmz) and len(qmz) > 0 and np.allclose(qmz, rmz, atol=2e-4) and np.allclose(qit, rit, atol=1e-3)) or \\\n                               entropy_similarity(qmz, qit, rmz, rit) >= DUP_SIM: dup = True; break\n                    if not dup: kept.append(x)\n                if len(kept) > MAX_REF:\n                    pri = np.array([0 if add_of_row[x] in M_by_add else 1 for x in kept]); kept = [kept[i] for i in np.argsort(pri, kind=\"stable\")[:MAX_REF]]\n                best = same = 0.0\n                for x in kept:\n                    (rmz, rit), rprec = ref_cache[x]\n                    for s in q:\n                        if s[\"mode\"] != mode_of_row[x]: continue\n                        sim = entropy_similarity(s[\"clean\"][0], s[\"clean\"][1], rmz, rit); best = max(best, sim)\n                        if add_of_row[x] == s[\"adduct\"]: same = max(same, sim)\n                lib[ci], lib_same[ci], nref[ci] = best, same, len(kept)\n            if len(ref_cache) > 200_000: ref_cache.clear()\n            # shortlist + official keys\n            fpz = np.nan_to_num(fpu, nan=np.nanmin(fpu) if np.isfinite(fpu).any() else 0.0); zf = (fpz - fpz.mean()) / (fpz.std() + 1e-9)\n            K = min(K_SHORT, n_u); top_fp = np.argsort(-np.nan_to_num(fpu, nan=-1e18), kind=\"stable\")[:K]; top_fpp = np.argsort(-(zf + SHORTLIST_LAM * pop), kind=\"stable\")[:K]\n            top_lib = [i for i in np.argsort(-lib, kind=\"stable\")[:K_LIB] if lib[i] > LIB_MIN]\n            short = np.union1d(np.union1d(top_fp, top_fpp), np.array(top_lib, dtype=np.int64))\n            s_smiles = [smiles[rep[keys[i]][\"i\"]] for i in short]\n            okeys = [k for rr_ in wp.map(_okey, [s_smiles[i:i + 25] for i in range(0, len(s_smiles), 25)]) for k in rr_]\n            best = {}\n            for j, i in enumerate(short):\n                if okeys[j] is not None: best.setdefault(okeys[j], []).append(i)\n            ppm_err = np.min(np.abs(np.array([[(M - m_) / m_ * 1e6 for m_ in cm_] for M in M_by_add.values()])), axis=0) if M_by_add else np.full(n_u, np.nan)\n            fp_sorted = np.sort(np.nan_to_num(fpu, nan=-1e18))\n            rows = []\n            for ok, idxs in best.items():\n                i = max(idxs, key=lambda j: np.nan_to_num(fpu[j], nan=-1e18))\n                rows.append(dict(key=ok, smiles=smiles[rep[keys[i]][\"i\"]], fp=float(fpu[i]) if np.isfinite(fpu[i]) else None, pop=float(pop[i]), lib=float(max(lib[j] for j in idxs)),\n                                 lib_same=float(max(lib_same[j] for j in idxs)), lib_nref=int(max(nref[j] for j in idxs)), in_train=bool(any(in_train[j] for j in idxs)),\n                                 in_tier=bool(any(trow[j] >= 0 for j in idxs)), in_coconut=bool(any(in_coco[j] for j in idxs)), ppm_err=float(np.nanmin([ppm_err[j] for j in idxs])),\n                                 fp_rank_window=int(len(fp_sorted) - np.searchsorted(fp_sorted, np.nan_to_num(fpu[i], nan=-1e18), side=\"left\")), **{n: float(comp[n][i]) for n in comp}))\n            feats[mid] = rows\n            molinfo[mid] = dict(n_query=len(q), ce_known=float(np.mean([s[\"ce\"] is not None for s in q])))\n            runner.append(dict(mid=mid, lib_max=float(lib.max()), spectra=[dict(mz=s[\"mz\"].tolist(), it=s[\"it\"].tolist(), prec=s[\"prec\"], adduct=s[\"adduct\"], mode=s[\"mode\"], instrument=s[\"instrument\"],\n                                                                                   ce_ev=s.get(\"ce_ev\"), ce_orig=s.get(\"ce_orig\"), ce_units=s.get(\"ce_units\")) for s in q], cands=[r[\"smiles\"] for r in rows if heavy_ok(r[\"smiles\"])]))\n            stats.append(dict(molecule=mid, n_query=len(q), adducts=\"|\".join(sorted(M_by_add)), n_window=len(smiles), n_unique=n_u, n_train=int(in_train.sum()), n_shortlist=len(rows), lib_max=float(lib.max()), seconds=time.time() - tm0))\n            if (n_i + 1) % 20 == 0 or n_i == len(mols) - 1:\n                d = pd.DataFrame(stats); log(f\"{n_i+1}/{len(mols)} | window median {d.n_unique.median():.0f} | shortlist median {d.n_shortlist.median():.0f} | lib_max>=0.6: {int((d.lib_max >= TAU).sum())} | {d.seconds.mean():.1f}s/mol\")\n    pd.DataFrame(stats).to_csv(f\"{P['work']}/b1_stage_stats.csv\", index=False)\n    try:   # per-candidate table (parity tests / debugging): molecule, official key, scores, provenance\n        pd.DataFrame([dict(molecule=m, **{k_: v_ for k_, v_ in r_.items()}) for m, rows_ in feats.items() for r_ in rows_]).to_parquet(f\"{P['work']}/b1_candidates.parquet\", index=False)\n    except Exception as e: log('candidate dump failed:', repr(e))\n    # ---- forward models: weakest library match first\n    runner.sort(key=lambda it: it[\"lib_max\"]); items = [dict(mid=it[\"mid\"], spectra=it[\"spectra\"], cands=it[\"cands\"]) for it in runner]\n    json.dump(items, open(f\"{P['work']}/b1_forward_in.json\", \"w\"))\n    GL, ICE = {}, {}\n    if not args.no_forward and items:\n        if KAGGLE:\n            sys.path.append(P[\"iceberg\"]); sys.path.append(P[\"glacier\"])\n            import fuse as ice_fuse, gl_fuse\n            try: preseed_site(P[\"iceberg\"], f\"{P['work']}/ice_site\")\n            except Exception as e: log(\"site preseed failed (runner will try its own install):\", repr(e))\n            try: ICE = ice_fuse.run_ice(P[\"iceberg\"], items, workdir=f\"{P['work']}/ice_work\", device=dev, budget_s=ICE_BUDGET, site=f\"{P['work']}/ice_site\", extra_args=[\"--any-rdkit\"]); log(\"ICE done\")\n            except Exception as e: log(\"ICE failed:\", repr(e))\n            try: GL = gl_fuse.run_gl(P[\"glacier\"], items, workdir=f\"{P['work']}/gl_work\", device=dev, budget_s=GL_BUDGET, site=f\"{P['work']}/ice_site\", wheels=os.path.join(P[\"iceberg\"], \"wheels\"), extra_args=[\"--any-rdkit\"]); log(\"GL done\")\n            except Exception as e: log(\"GL failed:\", repr(e))\n        else:\n            for name, pkg, script in ((\"gl\", P[\"glacier\"], \"gl_runner.py\"), (\"ice\", P[\"iceberg\"], \"ice_runner.py\")):\n                out = f\"{P['work']}/{name}_out.json\"\n                subprocess.run([P[\"runner_python\"], f\"{pkg}/{script}\", pkg, f\"{P['work']}/b1_forward_in.json\", out, \"--device\", dev, \"--site\", \"none\", \"--budget\", str(GL_BUDGET if name == \"gl\" else ICE_BUDGET)],\n                               stdout=open(f\"{P['work']}/{name}.log\", \"w\"), stderr=subprocess.STDOUT)\n                try: (GL if name == \"gl\" else ICE).update(json.load(open(out))); log(f\"{name} done\")\n                except Exception as e: log(f\"{name} output missing:\", repr(e))\n    # ---- fusion + submission\n    def z(x):\n        x = np.asarray(x, float); ok = np.isfinite(x); out = np.full_like(x, np.nan)\n        if ok.sum() >= 2 and x[ok].std(ddof=1) > 0: out[ok] = (x[ok] - x[ok].mean()) / x[ok].std(ddof=1)\n        elif ok.sum() >= 1: out[ok] = 0.0\n        out[~ok] = np.nanmin(out[ok]) if ok.any() else 0.0\n        return out\n    booster = None\n    model_path = args.rank_model or os.path.join(HERE, \"lambdarank.txt\")\n    if os.path.exists(model_path):\n        try:\n            import lightgbm as lgb\n            from b1_rankfeats import feature_matrix\n            booster = lgb.Booster(model_file=model_path); log(f\"learned fusion model loaded: {model_path} ({booster.num_trees()} trees)\")\n        except Exception as e:\n            booster = None; log(\"learned fusion unavailable -> fixed rule:\", repr(e))\n    sub, fused_stats = [], Counter()\n    for mid in all_ids:\n        rows = feats.get(str(mid), [])\n        if not rows: sub.append((mid, \"CCO\")); fused_stats[\"empty\"] += 1; continue\n        fp = np.array([r[\"fp\"] if r[\"fp\"] is not None else np.nan for r in rows]); pop = np.array([r[\"pop\"] for r in rows]); lib = np.array([r[\"lib\"] for r in rows])\n        g = np.array([(GL.get(str(mid)) or {}).get(r[\"smiles\"], np.nan) if (GL.get(str(mid)) or {}).get(r[\"smiles\"]) is not None else np.nan for r in rows], float)\n        ic = np.array([(ICE.get(str(mid)) or {}).get(r[\"smiles\"], np.nan) if (ICE.get(str(mid)) or {}).get(r[\"smiles\"]) is not None else np.nan for r in rows], float)\n        if booster is not None:\n            try:\n                mi = molinfo.get(str(mid), dict(n_query=1, ce_known=0.0))\n                X = feature_matrix(fp, g, ic, pop, [r[\"lsid\"] for r in rows], [r[\"lpmid\"] for r in rows], [r[\"lsynonyms\"] for r in rows], [r[\"np\"] for r in rows], [r[\"src_mask\"] for r in rows],\n                                   [r[\"lotus_refs\"] for r in rows], lib, [r[\"lib_same\"] for r in rows], [r[\"lib_nref\"] for r in rows], [r[\"in_tier\"] for r in rows], [r[\"in_train\"] for r in rows],\n                                   [r[\"in_coconut\"] for r in rows], [r[\"ppm_err\"] for r in rows], [r[\"fp_rank_window\"] for r in rows], mi[\"n_query\"], mi[\"ce_known\"])\n                order = np.argsort(-booster.predict(X), kind=\"stable\"); fused_stats[\"learned\"] += 1\n                sub.append((mid, \";\".join(rows[i][\"smiles\"] for i in order[:25]))); continue\n            except Exception as e:\n                fused_stats[\"learned_failed\"] += 1; log(\"learned fusion failed for\", mid, repr(e))\n        S = z(fp) + LAM * pop\n        if np.isfinite(g).sum() >= 2: S = S + W_GL * z(g); fused_stats[\"gl_used\"] += 1\n        if np.isfinite(ic).sum() >= 2: S = S + W_ICE * z(ic); fused_stats[\"ice_used\"] += 1\n        top = np.sort(lib)[::-1]; second = top[1] if len(top) > 1 else 0.0\n        if lib.max() >= TAU and lib.max() - second >= MARGIN: S = np.where(lib >= TAU, 100.0 + lib, S); fused_stats[\"lib_priority\"] += 1\n        order = np.argsort(-S, kind=\"stable\")\n        sub.append((mid, \";\".join(rows[i][\"smiles\"] for i in order[:25])))\n    df = pd.DataFrame(sub, columns=[\"molecule_id\", \"smiles\"]); df.to_csv(f\"{P['work']}/submission.csv\", index=False)\n    json.dump(dict(seconds=round(time.time() - T0), molecules=len(all_ids), scored=len(feats), fusion=dict(fused_stats), gl_mols=len(GL), ice_mols=len(ICE), learned_model=(model_path if booster is not None else None), params=dict(K=K_SHORT, W_GL=W_GL, W_ICE=W_ICE, LAM=LAM, TAU=TAU, MARGIN=MARGIN, folds=folds)),\n              open(f\"{P['work']}/b1_run_manifest.json\", \"w\"), indent=1)\n    log(f\"submission written: {df.shape}, fusion {dict(fused_stats)}\")\n    if args.solution:\n        sys.path.insert(0, os.path.join(os.path.dirname(os.path.dirname(HERE)), \"src\"))\n        import metric\n        sol = pd.read_csv(args.solution); sol[\"molecule_id\"] = sol[\"molecule_id\"].astype(str); sub_ = df.copy(); sub_[\"molecule_id\"] = sub_[\"molecule_id\"].astype(str)\n        sol = sol[sol.molecule_id.isin(set(sub_.molecule_id))]\n        official = float(metric.score(sol[[\"molecule_id\", \"smiles\"]].copy(), sub_[sub_.molecule_id.isin(set(sol.molecule_id))].copy(), \"molecule_id\"))\n        rr = {m: metric.per_molecule_reciprocal_rank(s_, str(p_).split(\";\")) for m, s_, p_ in zip(sol.molecule_id, sol.smiles, sub_.set_index(\"molecule_id\").loc[sol.molecule_id, \"smiles\"])}\n        per = sol.assign(rr=[rr[m] for m in sol.molecule_id]); per.to_csv(f\"{P['work']}/eval_per_molecule.csv\", index=False)\n        out_ = dict(official_mrr25=official, n=len(sol), masks=dict(rows=len(EXCL_ROWS), identities=len(EXCL_IDS)))\n        for col in (\"set\", \"group\", \"lib\"):\n            if col in per.columns: out_[f\"by_{col}\"] = {str(k): round(float(g.rr.mean()), 4) for k, g in per.groupby(col)}\n        json.dump(out_, open(f\"{P['work']}/eval_scores.json\", \"w\"), indent=1); log(f\"OFFICIAL MRR@25 = {official:.4f} on {len(sol)} molecules | {out_.get('by_set')}\")\n\n\nif __name__ == \"__main__\":\n    main()\n", "b1_rankfeats.py": "\"\"\"Shared candidate-feature construction for the learned fusion (training and inference use exactly this function).\n\nAll features are label-free and molecule-relative where it matters (z within the molecule's shortlist). Missing model scores are\nencoded explicitly (gl_missing / ice_missing) and the z-term is set to the list minimum (never 'average'); when a model has < 2 finite\nvalues on the list its z-term is 0 for every candidate.\n\"\"\"\nimport numpy as np\n\nFEATS = [\"z_fp\", \"z_gl\", \"z_ice\", \"fp\", \"gl\", \"ice\", \"pop_public\", \"lsid\", \"lpmid\", \"lsynonyms\", \"np_bit\", \"src_bit\", \"lotus\", \"lib_sim\", \"lib_same\", \"lib_nref\", \"lib_margin\",\n         \"lib_max_mol\", \"in_tier\", \"in_train\", \"in_coconut\", \"ppm_err\", \"fp_rank_window\", \"n_cands\", \"gl_missing\", \"ice_missing\", \"n_query\", \"ce_known\"]\n\n\ndef zlist(x):\n    x = np.asarray(x, float); ok = np.isfinite(x); out = np.full_like(x, np.nan)\n    if ok.sum() >= 2 and x[ok].std(ddof=1) > 0: out[ok] = (x[ok] - x[ok].mean()) / x[ok].std(ddof=1)\n    elif ok.sum() >= 1: out[ok] = 0.0\n    out[~ok] = np.nanmin(out[ok]) if ok.any() else 0.0\n    return out\n\n\ndef term(vals):\n    v = np.asarray(vals, float)\n    return zlist(v) if np.isfinite(v).sum() >= 2 else np.zeros(len(v))\n\n\ndef feature_matrix(fp, gl, ice, pop_public, lsid, lpmid, lsynonyms, np_bits, src_mask, lotus_refs, lib_sim, lib_same, lib_nref, in_tier, in_train, in_coconut,\n                   ppm_err, fp_rank_window, n_query, ce_known):\n    \"\"\"arrays of equal length (one molecule's candidates) -> (n, len(FEATS)) float matrix in FEATS order.\"\"\"\n    fp = np.asarray(fp, float); gl = np.asarray(gl, float); ice = np.asarray(ice, float); lib = np.asarray(lib_sim, float)\n    top = np.sort(lib)[::-1]; second = top[1] if len(top) > 1 else 0.0\n    cols = dict(z_fp=zlist(fp), z_gl=term(gl), z_ice=term(ice), fp=np.nan_to_num(fp, nan=np.nanmin(fp) if np.isfinite(fp).any() else 0.0),\n                gl=np.nan_to_num(gl, nan=-1.0), ice=np.nan_to_num(ice, nan=-1.0), pop_public=np.asarray(pop_public, float), lsid=np.asarray(lsid, float), lpmid=np.asarray(lpmid, float),\n                lsynonyms=np.asarray(lsynonyms, float), np_bit=(np.asarray(np_bits, float) != 0).astype(float), src_bit=((np.asarray(src_mask).astype(np.int64) & 1) != 0).astype(float),\n                lotus=np.log1p(np.maximum(np.asarray(lotus_refs, float), 0)), lib_sim=lib, lib_same=np.asarray(lib_same, float), lib_nref=np.asarray(lib_nref, float), lib_margin=lib - second,\n                lib_max_mol=np.full(len(fp), float(lib.max()) if len(lib) else 0.0), in_tier=np.asarray(in_tier, float), in_train=np.asarray(in_train, float), in_coconut=np.asarray(in_coconut, float),\n                ppm_err=np.nan_to_num(np.asarray(ppm_err, float), nan=10.0), fp_rank_window=np.log1p(np.asarray(fp_rank_window, float)), n_cands=np.full(len(fp), float(len(fp))),\n                gl_missing=(~np.isfinite(gl)).astype(float), ice_missing=(~np.isfinite(ice)).astype(float), n_query=np.full(len(fp), float(n_query)), ce_known=np.full(len(fp), float(ce_known)))\n    return np.column_stack([cols[c] for c in FEATS])\n"}
for name, text in SOURCES.items():
    open(f'/kaggle/working/b1/{name}', 'w', encoding='utf-8').write(text)
print('sources written', list(SOURCES), flush=True)
# official RDKit (scorer version) when provided by an attached dataset; otherwise keep the image's RDKit (keys may differ for ~1.6% of structures)
tag = f'cp{sys.version_info.major}{sys.version_info.minor}'
whl = [w for w in glob.glob('/kaggle/input/**/rdkit-2026.3.3-*.whl', recursive=True) if tag in w]
if whl:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '--no-index', '--quiet', whl[0]], capture_output=True, text=True)
    print('rdkit wheel install rc', r.returncode, r.stdout[-300:], r.stderr[-300:], flush=True)
else:
    print('no rdkit 2026.3.3 wheel attached; using the image RDKit', flush=True)
import rdkit; print('rdkit', rdkit.__version__, flush=True)


In [ ]:
import subprocess, sys, time, json, os
t0 = time.time()
cmd = [sys.executable, '-X', 'utf8', '/kaggle/working/b1/b1_pipeline.py', '--out', '/kaggle/working']
print('running', ' '.join(cmd), flush=True)
with open('/kaggle/working/b1_pipeline.log', 'w') as lf:
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True); lf.write(line)
    rc = p.wait()
print('pipeline rc', rc, f'{time.time()-t0:.0f}s', flush=True)
assert os.path.exists('/kaggle/working/submission.csv'), 'submission.csv missing'
import pandas as pd
sub = pd.read_csv('/kaggle/working/submission.csv'); samp = pd.read_csv('/kaggle/input/enveda-CASMI26-molecule-id-mass-spectra/sample_submission.csv')
assert len(sub) == len(samp) and set(sub.molecule_id) == set(samp.molecule_id), (len(sub), len(samp))
assert sub.smiles.notna().all() and sub.smiles.map(lambda s: len(str(s).split(';'))).max() <= 25
print(json.load(open('/kaggle/working/b1_run_manifest.json')))
print('OK', sub.shape, flush=True)
